# HR Attrition Analysis

This notebook shows a practical people analytics workflow to answer: Who is likely to leave and why?

We will use synthetic HR data covering monthly income, tenure, department, satisfaction, overtime, and job role.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

rng = np.random.default_rng(42)
n = 1500
departments = ['Sales', 'Research & Development', 'Human Resources']
roles = ['Sales Representative', 'Research Scientist', 'Laboratory Technician', 'Manager', 'Developer', 'HR Manager']
travel = ['Travel_Rarely', 'Travel_Frequently', 'Non_Travel']

df = pd.DataFrame({
    'Age': rng.integers(22, 60, size=n),
    'BusinessTravel': rng.choice(travel, size=n),
    'Department': rng.choice(departments, size=n),
    'DistanceFromHome': rng.integers(1, 30, size=n),
    'EducationField': rng.choice(['Life Sciences', 'Medical', 'Marketing', 'Technical Degree', 'Other'], size=n),
    'EnvironmentSatisfaction': rng.integers(1, 5, size=n),
    'JobRole': rng.choice(roles, size=n),
    'JobSatisfaction': rng.integers(1, 5, size=n),
    'MonthlyIncome': rng.integers(2000, 12000, size=n),
    'NumCompaniesWorked': rng.integers(1, 9, size=n),
    'OverTime': rng.choice(['Yes', 'No'], size=n),
    'YearsAtCompany': rng.integers(0, 30, size=n),
    'YearsSinceLastPromotion': rng.integers(0, 15, size=n),
})

z = -4.0 + 0.1 * df['DistanceFromHome'] + 0.9 * (df['OverTime'] == 'Yes') + 0.2 * df['NumCompaniesWorked'] + 0.3 * df['YearsSinceLastPromotion'] - 0.0007 * df['MonthlyIncome'] - 0.4 * df['JobSatisfaction'] + 0.3 * (df['Department'] == 'Sales')
prob = 1 / (1 + np.exp(-z))
df['Attrition'] = (rng.random(n) < prob).astype(int)
df.head()

## EDA: the why behind attrition

We begin with plots to identify relationships between turnover and employee characteristics.

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='OverTime', hue='Attrition')
plt.title('Attrition by Overtime Status')
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=df.groupby('Department')['Attrition'].mean().reset_index(), x='Department', y='Attrition', palette='viridis')
plt.title('Attrition Rate by Department')
plt.xticks(rotation=15)
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='Attrition', y='MonthlyIncome', palette='Set2')
plt.title('Monthly Income vs Attrition')
plt.show()

### Observations
- Employees working overtime appear to leave more often.
- Lower-paid employees show a higher attrition risk.
- Departments with strong sales pressure often display elevated turnover.

In [ ]:
X = df.drop(columns=['Attrition'])
y = df['Attrition']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

numeric = ['Age', 'DistanceFromHome', 'MonthlyIncome', 'NumCompaniesWorked', 'YearsAtCompany', 'YearsSinceLastPromotion']
categorical = ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'OverTime']
ordinal = ['EnvironmentSatisfaction', 'JobSatisfaction']

preprocess = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric),
    ('ord', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('enc', OrdinalEncoder())]), ordinal),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('enc', OneHotEncoder(handle_unknown='ignore'))]), categorical),
])

logreg = Pipeline([('preprocess', preprocess), ('model', LogisticRegression(max_iter=3000))])
rf = Pipeline([('preprocess', preprocess), ('model', RandomForestClassifier(n_estimators=300, random_state=42))])

logreg.fit(X_train, y_train)
rf.fit(X_train, y_train)

for name, model in [('Logistic Regression', logreg), ('Random Forest', rf)]:
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    print(name)
    print('Accuracy:', accuracy_score(y_test, pred))
    print('Precision:', precision_score(y_test, pred, zero_division=0))
    print('Recall:', recall_score(y_test, pred, zero_division=0))
    print('ROC AUC:', roc_auc_score(y_test, prob))
    print('---')

## Business interpretation

A good model is not only about prediction; it must explain action. If overtime, compensation, and promotion gaps dominate the prediction, the HR team can design focused retention strategies.